# RAG

In [ ]:
%pip install langchain-ollama langchain-community langchain-huggingface langchain-core langchain-experimental faiss-cpu huggingface-hub pypdf sentence-transformers

In [5]:
# Librerías estándar
import os

# LangChain
from langchain_ollama import OllamaLLM
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_experimental.text_splitter import SemanticChunker


## 1. Configuración del modelo LLM

In [6]:
llm = OllamaLLM(model="llama3.2:1b")

## 2. Carga de documentos

In [10]:
# Definimos la ruta de la carpeta
pdf_folder_path = "./data/"
docs = []

# Iteramos sobre los archivos en la carpeta 'data'
for file in os.listdir(pdf_folder_path):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder_path, file))
        docs.extend(loader.load())

print(f"✅ Se cargaron {len(docs)} páginas de los PDFs.")

✅ Se cargaron 6 páginas de los PDFs.


## 3. Configuración del modelo de embeddings

In [15]:
print("🔄 Descargando y cargando 'multilingual-e5-small'...")
print("Este modelo es ideal para español y eficiente en CPU/RAM.")

# Usamos el modelo E5 que es excelente para tareas de recuperación
embeddings_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small"
)

print("✅ Modelo de embeddings actualizado correctamente.")

🔄 Descargando y cargando 'multilingual-e5-small'...
Este modelo es ideal para español y eficiente en CPU/RAM.


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\Patricio Ricardi\PYTHON\LLM-RAG-Production\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Patricio Ricardi\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

✅ Modelo de embeddings actualizado correctamente.


## 4. División en fragmentos

In [16]:
# Ahora el Chunker usará la comprensión multilingüe de E5
text_splitter = SemanticChunker(
    embeddings_model, 
    breakpoint_threshold_type="percentile"
)

splits = text_splitter.split_documents(docs)

print(f"✅ Los PDFs se han vuelto a procesar con el nuevo modelo.")
print(f"✅ Total de fragmentos optimizados: {len(splits)}")

✅ Los PDFs se han vuelto a procesar con el nuevo modelo.
✅ Total de fragmentos optimizados: 15


In [17]:
# Verificamos la diferencia de tamaño entre fragmentos
if splits:
    print(f"\n📏 Tamaño del primer fragmento: {len(splits[0].page_content)} caracteres.")
    print("📄 Contenido del primer fragmento:")
    print("-" * 30)
    print(splits[0].page_content)


📏 Tamaño del primer fragmento: 1410 caracteres.
📄 Contenido del primer fragmento:
------------------------------
REPORTE  EXCLUSIVO:  La  NASA  Confirma  que  el  "Objeto  de  las  Marianas"  no  es  Natural  
ni
 
Humano
  Un  sumergible  no  tripulado  recuperó  en  la  Fosa  de  las  Marianas  un  artefacto  de  12.000  
años
 
de
 
antigüedad
 
que
 
emite
 
pulsos
 
electromagnéticos
 
sincronizados
 
con
 
la
 
rotación
 
de
 
la
 
Tierra. ---   Por  Javier  Ontiveros,  corresponsal  de  ciencia  y  tecnología    *Fecha  de  publicación:  3  de  mayo  de  2026  –  Última  actualización:  4  de  mayo  de  2026*   HONOLULU,  Hawái. –  En  una  conferencia  de  prensa  convocada  de  urgencia  ayer  a  las  
10:00
 
a.m. hora
 
local,
 
la
 
Administración
 
Nacional
 
de
 
Aeronáutica
 
y
 
el
 
Espacio
 
(NASA)
 
y
 
la
 
Oficina
 
de
 
Exploración
 
Oceánica
 
de
 
la
 
NOAA
 
revelaron
 
un
 
hallazgo
 
que,
 
según
 
sus
 
propias
 
palabras,
 
"no
 
tiene
 
precedentes
 
en
 

## 5. Almacén vectorial FAISS

In [18]:
%pip install flashrank

   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.9 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.9 MB 2.0 MB/s eta 0:00:06
   ---- ----------------------------------- 1.6/12.9 MB 2.2 MB/s eta 0:00:06
   ------ --------------------------------- 2.1/12.9 MB 2.3 MB/s eta 0:00:05
   -------- ------------------------------- 2.9/12.9 MB 2.4 MB/s eta 0:00:05
   ---------- ----------------------------- 3.4/12.9 MB 2.4 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.9 MB 2.6 MB/s eta 0:00:04
   --------------- ------------------------ 5.0/12.9 MB 2.8 MB/s eta 0:00:03
   ------------------- -------------------- 6.3/12.9 MB 3.1 MB/s eta 0:00:03
   ----------------------- ---------------- 7.6/12.9 MB 3.5 MB/s eta 0:00:02
   ---------------------------- ----------- 9.2/12.9 MB 3.8 MB/s eta 0:00:01
   ----------

In [ ]:
# 2. Creamos el almacén vectorial (Vector Store)
# Esto toma tus 'splits' de la Fase 3 y los convierte en vectores usando el modelo
print("🧠 Generando vectores e indexando en FAISS (esto puede tardar unos segundos)...")
vectorstore = FAISS.from_documents(splits, embeddings_model)

print("✅ Almacén vectorial creado exitosamente.")

# 3. Prueba rápida de búsqueda semántica (Retriever)
query = "¿Qué objeto encontró la NASA en las Marianas?"
docs_relacionados = vectorstore.similarity_search(query, k=2)

print("\n🔍 Prueba de búsqueda:")
for i, doc in enumerate(docs_relacionados):
    print(f"Resultado {i+1}: {doc.page_content[:100]}...")

## 6. Definición del pipeline RAG

In [ ]:
# 1. Definimos el Prompt
template = """Eres un asistente técnico experto. Responde la pregunta basándote únicamente en el siguiente contexto:
{context}

Pregunta: {question}

Respuesta:"""

prompt = ChatPromptTemplate.from_template(template)

# 2. Configuramos el flujo (Chain)
# Este pipeline: toma la pregunta -> busca en FAISS -> llena el prompt -> le pregunta a Llama
rag_chain = (
    {"context": vectorstore.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 7. Prueba final

In [ ]:
pregunta_final = "¿Cuales son los cuatro roles identificados en la colonia de Towada?"
print(f"Preguntando: {pregunta_final}\n")

respuesta_final = rag_chain.invoke(pregunta_final)
print("--- RESPUESTA DEL RAG ---")
print(respuesta_final)